In [1]:
import rioxarray as rx
import geopandas as gp
import shapely as shp
import pyproj
import numpy as np
import pycountry
import json
import urllib.request

from emu_renewal.inputs import DATA_PATH

# Stop GeoPandas warning about area intersection calculations
import warnings
warnings.filterwarnings("ignore", "Geometry is in a geographic CRS")

In [2]:
country = "France"
iso3 = pycountry.countries.lookup(country).alpha_3

# Usually level 1 for European countries and 2 for other, but check in Facebook movement table
gadm_level = 1

In [3]:
def raster_to_polydf(raster_ds, data_name):
    pix_buf = (raster_ds.coords["x"][1] - raster_ds.coords["x"][0]).data * 0.5
    geoms = []
    nodata_mask = raster_ds.rio.nodata

    data = raster_ds.data[0]
    n_valid = (data != nodata_mask).sum()
    out_data = np.empty(n_valid)
    valid_idx = 0

    for ix, x in enumerate(raster_ds.coords["x"].data):
        for iy, y in enumerate(raster_ds.coords["y"].data):
            cell_data = data[iy, ix]
            if cell_data != nodata_mask:
                geoms.append(shp.Polygon([
                        (x - pix_buf, y - pix_buf),
                        (x + pix_buf, y - pix_buf),
                        (x + pix_buf, y + pix_buf),
                        (x - pix_buf, y + pix_buf),
                    ]
                ))
                out_data[valid_idx] = cell_data
                valid_idx += 1
            
    data = data.flatten()
    
    return gp.GeoDataFrame({data_name: out_data}, geometry=geoms)

In [4]:
# Gridded Population of the World 30sec 2020 dataset
# https://www.earthdata.nasa.gov/data/projects/gpw
pop_ds = rx.open_rasterio(DATA_PATH / "population/gpw_v4_population_count_rev11_2020_30_sec.tif")

# Alternative dataset from:
# https://github.com/lulingliu/GlobPOP
#pop_ds = rx.open_rasterio(DATA_PATH / "population/GlobPOP_Count_30arc_2020_I32.tiff")

In [5]:
# Download the appropriate GADM boundaries json
source = f"https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3}_{gadm_level}.json.zip"
dest = DATA_PATH / f"population/gadm_input_json/gadm41_{iso3}_{gadm_level}.json.zip"
urllib.request.urlretrieve(source, dest)

# GADM administrative boundaries as obtained from:
# https://gadm.org/download_country.html

# These files are directly downloadable as of 21/01/2025 via
# https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3_code}_{gadm_level}.json.zip

poly_df = gp.read_file(dest)

In [6]:
# Set up the dimensions of a population cell
pix_dim = float((pop_ds.coords["x"][1] - pop_ds.coords["x"][0]).data)

pop_dict = {}

# Loop over polygons relevant to the country of interest
for ploc, poly_id in enumerate(poly_df[f"GID_{gadm_level}"]):

    # Get the relevant gridded population data, ensuring correct projection
    pop_bounds = np.array(poly_df.iloc[ploc].geometry.bounds)
    expanded_bounds = pop_bounds + np.array(([-pix_dim, -pix_dim, pix_dim, pix_dim]))
    dsc = pop_ds.rio.clip_box(*expanded_bounds)
    pop_df = raster_to_polydf(dsc, "population")
    pop_df = pop_df.set_crs(pyproj.CRS.from_wkt(pop_ds.rio.crs.to_wkt()))

    # Find population based on intersections
    isect = gp.overlay(pop_df, poly_df.iloc[ploc: ploc + 1], keep_geom_type=False)
    pop_val = float((isect.area / pix_dim ** 2.0 * isect.population).sum())
    pop_dict[poly_id] = pop_val
    print(f"{poly_id} has population of {round(pop_val / 1e3, 3)} thousand")

FRA.1_1 has population of 8300.427 thousand
FRA.2_1 has population of 2949.236 thousand
FRA.3_1 has population of 3494.23 thousand
FRA.4_1 has population of 2680.351 thousand
FRA.5_1 has population of 359.378 thousand
FRA.6_1 has population of 5766.409 thousand
FRA.7_1 has population of 6067.935 thousand
FRA.8_1 has population of 12763.522 thousand
FRA.9_1 has population of 3438.822 thousand
FRA.10_1 has population of 6303.121 thousand
FRA.11_1 has population of 6345.711 thousand
FRA.12_1 has population of 3971.041 thousand
FRA.13_1 has population of 5291.152 thousand


In [7]:
json.dump(pop_dict, open(DATA_PATH / f"population/gadm_est/{iso3}_{gadm_level}.json", "w"))

In [8]:
f"Total population for country is {round(sum(pop_dict.values()) / 1e6, 3)} million"

'Total population for country is 67.731 million'